
# Fundamentos de Robótica: Cinemática y Orientación Espacial 🤖

<a href="https://colab.research.google.com/github/Reve7339/robotics-foundations/blob/main/Representation_of_Orientation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este Notebook interactivo presenta los conceptos clave para el modelado de robots manipuladores de forma clara y equilibrada. A lo largo del documento, combinaremos la teoría estándar del curso con visualizaciones 3D interactivas en `plotly`, matemáticas formales en LaTeX y código práctico en Python.

Antes de comenzar, instala y carga las librerías necesarias ejecutando la siguiente celda:


In [ ]:

%pip install numpy scipy plotly -q
import numpy as np
import plotly.graph_objects as go
from scipy.spatial.transform import Rotation as R
import math
print("✅ Entorno listo.")



---
## 1. Introducción a la Cinemática de Manipuladores

La **cinemática** en robótica es el estudio del movimiento geométrico de un robot, sin tomar en cuenta las fuerzas que lo producen. Es esencialmente el "diccionario" que traduce entre dos espacios distintos:

* **Espacio Articular (Joint Space):** Son los ángulos físicos de los motores o articulaciones del robot. Es el lenguaje que entiende el robot internamente.
* **Espacio Cartesiano (Task Space):** Son las coordenadas $(X, Y, Z)$ y la orientación en el espacio tridimensional. Es el lenguaje en el que los ingenieros solemos definir las tareas.

Existen dos procesos analíticos principales:
* **Cinemática Directa:** Calcular la posición y orientación final a partir de los ángulos articulares.
* **Cinemática Inversa:** Calcular los ángulos articulares necesarios para alcanzar una posición objetivo.

A continuación, visualizamos el cálculo de la cinemática directa para un brazo planar simple de 2 grados de libertad:


In [ ]:

def cinematica_brazo_2d(q1_deg, q2_deg):
    l1, l2 = 1.0, 1.0
    p0 = np.array([0, 0])
    q1 = np.radians(q1_deg)
    p1 = p0 + np.array([l1 * np.cos(q1), l1 * np.sin(q1)])
    q12 = np.radians(q1_deg + q2_deg)
    p2 = p1 + np.array([l2 * np.cos(q12), l2 * np.sin(q12)])
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[p0[0], p1[0], p2[0]], y=[p0[1], p1[1], p2[1]], mode='lines+markers+text',
                             text=["Base", "Articulación 1", "Efector Final"], textposition="bottom right",
                             marker=dict(size=10, color=['black', 'blue', 'red']), line=dict(width=4, color='gray')))
    
    fig.update_layout(title="Cinemática Directa (Brazo Planar)", xaxis=dict(range=[-2.5, 2.5], title='Eje X (m)'),
                      yaxis=dict(range=[-2.5, 2.5], title='Eje Y (m)', scaleanchor="x", scaleratio=1), showlegend=False)
    fig.show()

cinematica_brazo_2d(q1_deg=45, q2_deg=-30)



---
## 2. Matrices de Rotación (Rotation Matrix)

Para describir hacia dónde "mira" un objeto (su orientación), utilizamos una **Matriz de Rotación**. Esta matriz de $3 \times 3$ contiene tres vectores columna que indican cómo se alinean los ejes locales X, Y y Z del objeto respecto al mundo inercial.

Aunque la matriz usa 9 valores, sus columnas deben tener longitud 1 y ser perpendiculares entre sí (6 restricciones geométricas de ortogonalidad). Por tanto, la matriz tiene **3 grados de libertad** matemáticos, denotándose como perteneciente al grupo especial ortogonal $SO(3)$.

### Matrices de Rotación Elementales
Cualquier rotación en el espacio se puede descomponer o construir multiplicando rotaciones básicas alrededor de los ejes inerciales principales. Para un ángulo $\theta$, las matrices elementales se definen analíticamente como:

**1. Rotación sobre el eje X ($R_x$):**
$$ R_x(\theta) = \begin{bmatrix} 1 & 0 & 0 \\ 0 & \cos\theta & -\sin\theta \\ 0 & \sin\theta & \cos\theta \end{bmatrix} $$

**2. Rotación sobre el eje Y ($R_y$):**
$$ R_y(\theta) = \begin{bmatrix} \cos\theta & 0 & \sin\theta \\ 0 & 1 & 0 \\ -\sin\theta & 0 & \cos\theta \end{bmatrix} $$

**3. Rotación sobre el eje Z ($R_z$):**
$$ R_z(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix} $$

A continuación, puedes graficar y visualizar dinámicamente cómo actúa cualquiera de estas tres rotaciones elementales sobre un sistema de referencia:


In [ ]:

def graficar_rotacion_elemental(eje='z', angulo_grados=45):
    # Validar el eje ingresado
    eje = eje.lower()
    if eje not in ['x', 'y', 'z']:
        print("Error: El eje debe ser 'x', 'y', o 'z'.")
        return
        
    # Generar matriz
    r = R.from_euler(eje, angulo_grados, degrees=True)
    matriz_R = r.as_matrix()
    
    frame_inercial = np.eye(3)
    frame_cuerpo = matriz_R @ frame_inercial
    
    fig = go.Figure()
    
    def plot_basis(vectors, color_x, color_y, color_z, name_prefix, width=4):
        axes = ['X', 'Y', 'Z']
        colors = [color_x, color_y, color_z]
        for i, vec in enumerate(vectors):
            name = f"{axes[i]} {name_prefix}"
            fig.add_trace(go.Scatter3d(x=[0, vec[0]], y=[0, vec[1]], z=[0, vec[2]],
                                       mode='lines', line=dict(color=colors[i], width=width), name=name))
            fig.add_trace(go.Scatter3d(x=[vec[0]], y=[vec[1]], z=[vec[2]],
                                       mode='markers+text', marker=dict(size=3, color=colors[i]),
                                       text=[name], textposition='top center', showlegend=False))
        
    # Colores estandarizados para RGB (X=Red, Y=Green, Z=Blue) pero pálidos para el inercial
    plot_basis(frame_inercial, 'lightcoral', 'lightgreen', 'lightblue', 'Mundo', width=2)
    # Colores fuertes para el marco rotado
    plot_basis(frame_cuerpo, 'red', 'green', 'blue', 'Rotado')
    
    fig.update_layout(title=f"Rotación Elemental R_{eje}({angulo_grados}°)",
                      scene=dict(xaxis=dict(range=[-1.5,1.5]), yaxis=dict(range=[-1.5,1.5]), zaxis=dict(range=[-1.5,1.5]), aspectmode='cube'))
    fig.show()

# Puedes cambiar el eje a 'x', 'y' o 'z', y el ángulo que desees
graficar_rotacion_elemental(eje='y', angulo_grados=60)



---
## 3. Representaciones Mínimas y Ángulos de Euler

Dado que la orientación tiene solo 3 grados de libertad, matemáticamente es posible representarla usando estrictamente 3 parámetros. La convención analítica más famosa son los **Ángulos de Euler**.

Este método parametriza una orientación componiendo sucesivamente tres rotaciones elementales (como las descritas arriba). En robótica móvil y aeronáutica, usamos casi siempre la convención **ZYX** o **Roll, Pitch, Yaw** (Alabeo, Cabeceo y Guiñada):
- **Yaw ($\psi$):** Guiñada direccional (eje Z).
- **Pitch ($\theta$):** Cabeceo vertical (eje Y).
- **Roll ($\phi$):** Alabeo lateral (eje X).

### Composición Matemática
Para obtener la matriz de rotación final de la secuencia completa, multiplicamos las matrices elementales. Si realizamos las rotaciones relativas a ejes fijos, el producto matricial se efectúa multiplicando de derecha a izquierda:

$$ R_{ZYX}(\psi, \theta, \phi) = R_z(\psi) \cdot R_y(\theta) \cdot R_x(\phi) $$

Al desarrollar algebraicamente esa multiplicación (multiplicando matricialmente las rotaciones elementales de la sección 2), obtenemos la matriz general $R$ paramétrica:

$$
R = \begin{bmatrix} 
c_\psi c_\theta & c_\psi s_\theta s_\phi - s_\psi c_\phi & c_\psi s_\theta c_\phi + s_\psi s_\phi \\
s_\psi c_\theta & s_\psi s_\theta s_\phi + c_\psi c_\phi & s_\psi s_\theta c_\phi - c_\psi s_\phi \\
-s_\theta & c_\theta s_\phi & c_\theta c_\phi 
\end{bmatrix}
$$
*(Donde $c$ denota coseno y $s$ denota seno).*

### Singularidades: El "Gimbal Lock"
La principal vulnerabilidad de usar solo 3 parámetros es analítica. Observa la matriz $R$ arriba: si el Pitch $\theta = 90^\circ$ o $\pi/2$, entonces $\cos(\theta) = c_\theta = 0$. Muchos términos desaparecen o se fusionan y las variables Yaw ($\psi$) y Roll ($\phi$) ya no pueden distinguirse entre sí (quedan atrapadas en términos de sumas $s_{\psi+\phi}$). 
Físicamente, el eje de Yaw inicial y el eje de Roll final han quedado alineados colinealmente. Se ha perdido un grado de libertad efectivo en el mecanismo.

Visualicemos cómo se compone una rotación por pasos de Euler:


In [ ]:

def graficar_euler_secuencial(yaw, pitch, roll):
    # Creamos las 3 secuencias paso a paso
    # Paso 0: Identidad
    R0 = np.eye(3)
    
    # Paso 1: Solo Yaw (Z)
    R1 = R.from_euler('z', yaw, degrees=True).as_matrix()
    
    # Paso 2: Yaw (Z) * Pitch (Y)
    R2 = R.from_euler('zy', [yaw, pitch], degrees=True).as_matrix()
    
    # Paso 3: Yaw (Z) * Pitch (Y) * Roll (X)
    R3 = R.from_euler('zyx', [yaw, pitch, roll], degrees=True).as_matrix()
    
    fig = go.Figure()
    
    # Dibujamos las 3 tramas paso a paso para ver el viaje del marco de coordenadas
    def plot_frame(mat, title, opacity=1.0):
        c = mat @ np.eye(3)
        axes = ['X', 'Y', 'Z']
        colors = ['red', 'green', 'blue']
        for i, vec in enumerate(c):
            fig.add_trace(go.Scatter3d(x=[0, vec[0]], y=[0, vec[1]], z=[0, vec[2]],
                                       mode='lines+text', line=dict(color=colors[i], width=3),
                                       text=[None, f"{axes[i]}_{title}"], textfont=dict(color=colors[i]),
                                       opacity=opacity, showlegend=False))
            
    plot_frame(R0, "Mundo", opacity=0.2)
    plot_frame(R1, "1(Yaw)", opacity=0.4)
    plot_frame(R2, "2(Pitch)", opacity=0.6)
    plot_frame(R3, "Final(Roll)", opacity=1.0)

    fig.update_layout(title=f"Secuencia Euler ZYX: Yaw={yaw}°, Pitch={pitch}°, Roll={roll}°",
                      scene=dict(xaxis=dict(range=[-1.5,1.5]), yaxis=dict(range=[-1.5,1.5]), zaxis=dict(range=[-1.5,1.5]), aspectmode='cube'))
    fig.show()

# Observemos cómo el sistema "viaja" por 3 orientaciones intermedias para llegar a la final
graficar_euler_secuencial(yaw=45, pitch=45, roll=45)



---
## 4. Representación de Eje y Ángulo (Angle and Axis)

El **Teorema de Rotación de Euler** establece que cualquier desplazamiento espacial de un cuerpo rígido puede expresarse analíticamente como un único giro de magnitud $\vartheta$ alrededor de un solo eje tridimensional unitario $\mathbf{r} = [r_x, r_y, r_z]^T$.

### Fórmula de Rotación de Rodrigues
La teoría de grupos ortogonales de Lie provee el mapeo directo entre un vector de Eje-Ángulo y su correspondiente Matriz de Rotación en $SO(3)$ a través de la célebre **Fórmula de Rodrigues**:

$$ R = I + \sin(\vartheta) S(\mathbf{r}) + (1 - \cos(\vartheta)) S^2(\mathbf{r}) $$

Donde $I$ es la matriz identidad $3\times3$ y $S(\mathbf{r})$ denota la matriz antisimétrica (skew-symmetric operator) construida a partir de las componentes del eje $\mathbf{r}$:

$$
S(\mathbf{r}) = \begin{bmatrix} 0 & -r_z & r_y \\ r_z & 0 & -r_x \\ -r_y & r_x & 0 \end{bmatrix}
$$

### Cálculo del Ángulo y Eje (El mapeo inverso)
Si por el contrario se cuenta con una matriz de rotación $R$ y se desea extraer el ángulo de giro equivalente, se recurre a la traza de la matriz (la suma de los elementos de su diagonal principal, que es un invariante geométrico):

$$ \vartheta = \arccos\left(\frac{\text{Traza}(R) - 1}{2}\right) $$
*(Siempre que $\vartheta \in (0, \pi)$).*

A continuación, una visualización mejorada que muestra claramente la relación ortogonal de un punto pivotando alrededor del eje invariante:


In [ ]:

def graficar_eje_angulo(eje_vector, angulo_deg):
    eje_norm = np.array(eje_vector) / np.linalg.norm(eje_vector)
    matriz_R = R.from_rotvec(np.radians(angulo_deg) * eje_norm).as_matrix()
    
    fig = go.Figure()
    
    # Eje de rotación principal r
    fig.add_trace(go.Scatter3d(x=[-eje_norm[0]*1.5, eje_norm[0]*1.5], y=[-eje_norm[1]*1.5, eje_norm[1]*1.5], z=[-eje_norm[2]*1.5, eje_norm[2]*1.5],
                               mode='lines', line=dict(color='orange', width=8), name='Eje Invariante r'))
    
    # Geometría de un punto rotando
    p_0 = np.array([1, 0, 0])
    p_1 = matriz_R @ p_0
    
    # Calculamos la proyección ortogonal del punto sobre el eje para graficar el radio de giro
    proj_dist = np.dot(p_0, eje_norm)
    proj_point = proj_dist * eje_norm
    
    # Graficar el centro de rotación (proyección en el eje) a p_0 y p_1
    fig.add_trace(go.Scatter3d(x=[proj_point[0], p_0[0]], y=[proj_point[1], p_0[1]], z=[proj_point[2], p_0[2]],
                               mode='lines', line=dict(color='gray', width=2, dash='dot'), name='Radio inicial'))
    fig.add_trace(go.Scatter3d(x=[proj_point[0], p_1[0]], y=[proj_point[1], p_1[1]], z=[proj_point[2], p_1[2]],
                               mode='lines', line=dict(color='red', width=2, dash='dot'), name='Radio final'))
    
    # Graficar los puntos
    fig.add_trace(go.Scatter3d(x=[p_0[0]], y=[p_0[1]], z=[p_0[2]], mode='markers+text', text=['p_0 (Inicial)'], marker=dict(size=6, color='black'), name='p_0'))
    fig.add_trace(go.Scatter3d(x=[p_1[0]], y=[p_1[1]], z=[p_1[2]], mode='markers+text', text=['p_1 (Rotado)'], marker=dict(size=6, color='red'), name='p_1'))
                               
    fig.update_layout(title=f"Rotación de Rodrigues: Eje r={np.round(eje_norm, 2)}^T, Ángulo={angulo_deg}°",
                      scene=dict(xaxis=dict(range=[-1.5,1.5]), yaxis=dict(range=[-1.5,1.5]), zaxis=dict(range=[-1.5,1.5]), aspectmode='cube'))
    fig.show()

# Probemos con un eje diagonal arbitrario y 90 grados
graficar_eje_angulo(eje_vector=[1, 1, 1], angulo_deg=90)



---
## 5. Representación de Cuatro Parámetros (Cuaterniones Unitarios)

Para evitar el problema del "Gimbal Lock" sin tener que lidiar con los 9 parámetros de la matriz de rotación, la robótica moderna utiliza los **Cuaterniones Unitarios**.

Un cuaternión es una extensión de los números complejos ($Q = \eta + \epsilon_x\hat{i} + \epsilon_y\hat{j} + \epsilon_z\hat{k}$) que consta de una parte escalar ($\eta$) y un vector tridimensional ($\mathbf{\epsilon}$). 

### Mapeo analítico desde Eje-Ángulo
Se relacionan directamente con el Teorema de Euler. Si tenemos un eje de rotación unitario $\mathbf{r}$ y un ángulo $\vartheta$, las componentes del cuaternión se definen analíticamente como:
$$ \eta = \cos\left(\frac{\vartheta}{2}\right) $$
$$ \mathbf{\epsilon} = \sin\left(\frac{\vartheta}{2}\right)\mathbf{r} $$

Al imponerles la **condición de norma unitaria**:
$$ \eta^2 + \epsilon_x^2 + \epsilon_y^2 + \epsilon_z^2 = 1 $$
logran representar orientaciones de manera robusta globalmente. 

Además, los microcontroladores procesan los cuaterniones extremadamente rápido mediante el producto hamiltoniano, ya que dependen de multiplicaciones algebraicas en lugar de pesadas funciones trascendentes de trigonometría.


In [ ]:

def graficar_rotacion_cuaternion(eje_vector, angulo_deg):
    eje_norm = np.array(eje_vector) / np.linalg.norm(eje_vector)
    
    # Calcular cuaternión matemáticamente usando las fórmulas explícitas
    theta_rad = np.radians(angulo_deg)
    eta = np.cos(theta_rad / 2)
    epsilon = np.sin(theta_rad / 2) * eje_norm
    q_math = [epsilon[0], epsilon[1], epsilon[2], eta] # Formato Scipy (x, y, z, w)
    
    # Instanciar el objeto de rotación de Scipy directamente desde el Cuaternión
    r = R.from_quat(q_math)
    matriz_R = r.as_matrix()
    
    print(f"1. Cuaternión analítico calculado: eta={np.round(eta, 3)}, epsilon={np.round(epsilon, 3)}")
    print(f"2. Cuaternión extraído de Scipy:  {np.round(r.as_quat(), 3)} (formato x,y,z,w)")
    
    # Generar un pequeño cubo (8 vértices)
    v = np.array([[-1,-1,-1], [1,-1,-1], [1,1,-1], [-1,1,-1], [-1,-1,1], [1,-1,1], [1,1,1], [-1,1,1]]) * 0.5
    
    # Rotar el cubo usando la matriz equivalente obtenida del cuaternión
    v_rot = (matriz_R @ v.T).T
    
    fig = go.Figure()
    
    # Función auxiliar para dibujar las aristas del cubo
    def draw_cube(vertices, color, name):
        edges = [(0,1), (1,2), (2,3), (3,0), (4,5), (5,6), (6,7), (7,4), (0,4), (1,5), (2,6), (3,7)]
        for i, edge in enumerate(edges):
            show_leg = True if i == 0 else False
            fig.add_trace(go.Scatter3d(x=[vertices[edge[0]][0], vertices[edge[1]][0]],
                                       y=[vertices[edge[0]][1], vertices[edge[1]][1]],
                                       z=[vertices[edge[0]][2], vertices[edge[1]][2]],
                                       mode='lines', line=dict(color=color, width=4), name=name, showlegend=show_leg))
            
    draw_cube(v, 'gray', 'Cubo Inicial')
    draw_cube(v_rot, 'blue', 'Cubo Rotado')
    
    fig.update_layout(title=f"Rotación Cuaterniónica en 3D (Eje={np.round(eje_norm,1)}, Ángulo={angulo_deg}°)",
                      scene=dict(xaxis=dict(range=[-1.5,1.5]), yaxis=dict(range=[-1.5,1.5]), zaxis=dict(range=[-1.5,1.5]), aspectmode='cube'))
    fig.show()

# Probemos rotar un cuerpo tridimensional 45 grados sobre el eje Z usando nuestro cuaternión
graficar_rotacion_cuaternion(eje_vector=[0, 0, 1], angulo_deg=45)



---
## 6. Casos Prácticos Aplicados

Veamos cómo se aplican estas teorías matemáticas a problemas concretos en la robótica y la navegación.

### 6.1 Posicionamiento Relativo (El Robot y la Cámara)
**Problema:** Una cámara está montada firmemente en la herramienta de un brazo robótico. Sabemos que la herramienta está rotada **90° sobre el eje Z** respecto a la base del robot. 
La cámara detecta una pieza a una coordenada local de $(x=1, y=0, z=2)$ metros. 
**Calculemos en qué coordenada respecto a la base general del robot se encuentra realmente la pieza.**


In [ ]:

# Matriz de rotación de la herramienta (cámara) respecto a la base
R_camara_base = R.from_euler('z', 90, degrees=True).as_matrix()

# Coordenada local de la pieza (vista por la cámara)
punto_camara = np.array([1, 0, 2])

# Transformación de coordenadas
punto_base = R_camara_base @ punto_camara

print(f"Posición detectada localmente por la cámara: {punto_camara}")
print(f"Posición absoluta de la pieza en el mundo: {np.round(punto_base, 2)}")
print("Nota: El vector se rota correctamente hacia el sistema de coordenadas inercial base.")



### 6.2 El Peligro del Gimbal Lock en la Navegación (Drones)
**Problema:** Programamos la orientación de un dron ingresando comandos directos de Euler (Yaw, Pitch, Roll).
Le ordenamos orientarse en:
1. Yaw = 45°
2. Pitch = 90° (nariz apuntando directamente al cielo)
3. Roll = -45°

**Evaluemos analíticamente el resultado para visualizar el "Gimbal Lock".** 

> **Nota:** Al ejecutar el código de abajo, verás que la propia librería `scipy` arroja un mensaje rojo (`UserWarning: Gimbal lock detected`). ¡Esto no es un error de tu código, es la librería confirmando matemáticamente que el dron ha entrado en una singularidad y que es imposible calcular los ángulos de manera unívoca!


In [ ]:

# Estado de orientación deseado
yaw_deseado, pitch_deseado, roll_deseado = 45, 90, -45
rotacion_dron = R.from_euler('zyx', [yaw_deseado, pitch_deseado, roll_deseado], degrees=True)

print(f"Comandos teóricos ingresados: Yaw={yaw_deseado}°, Pitch={pitch_deseado}°, Roll={roll_deseado}°")

# Le preguntamos a Scipy cómo interpreta realmente esta rotación
angulos_reales = rotacion_dron.as_euler('zyx', degrees=True)
print(f"Estado real interpretado: Yaw={np.round(angulos_reales[0])}°, Pitch={np.round(angulos_reales[1])}°, Roll={np.round(angulos_reales[2])}°")

print("\nDiagnóstico de Gimbal Lock:")
print("Al estar el Pitch en 90°, el eje inercial de Yaw (Z_0) y el eje local de Roll (X_2) han quedado colineales.")
print("Por lo tanto, la suma de las rotaciones se acopla, resultando matemáticamente en un Yaw de 0 y un Roll de 0.")



### 6.3 Interpolación de Trayectorias Suaves (Cuaterniones - SLERP)
**Problema:** Necesitamos que un brazo robótico mueva su soldador desde una orientación inicial recta hasta una orientación oblicua ($Yaw=45°, Pitch=60°, Roll=30°$). 
Si interpolamos directamente usando matrices o Euler, el movimiento derivativo puede ser discontinuo y dañino para los motores. 
**Solución:** Utilizar SLERP (*Spherical Linear Interpolation*) con cuaterniones para generar cuadros intermedios (waypoints) sobre la variedad espacial que garanticen un perfil de velocidad angular (ω) uniforme.


In [ ]:

from scipy.spatial.transform import Slerp

# Orientaciones límite
rot_A = R.from_euler('xyz', [0, 0, 0], degrees=True)
rot_B = R.from_euler('xyz', [45, 60, 30], degrees=True)

# Interpolador SLERP
tiempos_frontera = [0, 1]
rotaciones_clave = R.concatenate([rot_A, rot_B])
slerp = Slerp(tiempos_frontera, rotaciones_clave)

# Generación de 5 puntos clave en el tiempo (t)
t_pasos = np.linspace(0, 1, 5)
rotaciones_suaves = slerp(t_pasos)

print("Progresión del cuaternión unitario (x, y, z, w) durante la interpolación:")
for t, rot in zip(t_pasos, rotaciones_suaves):
    q = np.round(rot.as_quat(), 4)
    print(f"Iteración t={t*100:3.0f}% -> Q(t) = {q}")
    
print("\nObservación: La interpolación geodésica en el espacio de cuaterniones asegura la continuidad suave requerida por el controlador de los servomotores.")
